# User Data Processing and Analysis

This notebook processes user data files, compares the processed data against ground truth labels, and calculates the Word Error Rate (WER) for two datasets, `X` and `Y`. The analysis includes data loading, preprocessing, validation, and accuracy computation.


In [1]:
import os
import numpy as np

## Constants and Parameters

- **DATA_DIRECTORY:** Path to the directory containing user data files.
- **GROUND_TRUTH_LABELS:** List of ground truth labels for comparison.


In [2]:
# Directory containing user data files
DATA_DIRECTORY = "userData"

# Ground truth labels for comparison
GROUND_TRUTH_LABELS = [
    8, 4, 7, 4, 8, 0, 7, 1, 2, 6,
    8, 2, 3, 8, 0, 8, 0, 4, 0, 2,
    5, 1, 9, 0, 2, 0, 2, 4, 7, 2,
    6, 2
]


## Data Processing

Define a function to process each user data file. The function reads the file, splits its content into two parts using the delimiter `"==="`, cleans and parses the data into triplets `[a, b, c]`, and handles non-integer values by assigning a default value of `-1`.


In [3]:
def process_file(file_path, filename):
    """
    Processes a single user data file.

    Parameters:
        file_path (str): Path to the user data file.
        filename (str): Name of the file (for logging purposes).

    Returns:
        tuple: Two lists containing processed X and O data.
    """
    with open(file_path, "r") as file:
        data_parts = file.read().split("===")
        
        if len(data_parts) < 2:
            print(f"Warning: File '{filename}' does not contain '===' delimiter.")
            return None, None
        
        # Process first part (X data)
        x_raw = data_parts[0].replace(",", " ").replace("-", " ").split()
        x_processed = []
        
        for i in range(0, len(x_raw), 3):
            if i + 2 >= len(x_raw):
                print(f"Warning: Incomplete triplet in file '{filename}' for X data.")
                break
            
            a_str, b_str, c_str = x_raw[i:i+3]
            
            try:
                a = int(a_str)
            except ValueError:
                a = -1
                print(f"Info: Non-integer value '{a_str}' encountered in X data. Set to -1.")
            
            try:
                b = int(b_str)
                c = int(c_str)
            except ValueError:
                b = c = -1
                print(f"Warning: Non-integer values encountered in X data for triplet starting with '{a_str}'. Set to -1.")
            
            x_processed.append([a, b, c])
        
        # Process second part (O data)
        o_raw = data_parts[1].replace(",", " ").replace("-", " ").split()
        o_processed = []
        
        for i in range(0, len(o_raw), 3):
            if i + 2 >= len(o_raw):
                print(f"Warning: Incomplete triplet in file '{filename}' for O data.")
                break
            
            a_str, b_str, c_str = o_raw[i:i+3]
            
            try:
                a = int(a_str)
            except ValueError:
                a = -1
                print(f"Info: Non-integer value '{a_str}' encountered in O data. Set to -1.")
            
            try:
                b = int(b_str)
                c = int(c_str)
            except ValueError:
                b = c = -1
                print(f"Warning: Non-integer values encountered in O data for triplet starting with '{a_str}'. Set to -1.")
            
            o_processed.append([a, b, c])
    
    return x_processed, o_processed


## Loading and Processing Data Files

Iterate through each file in the `DATA_DIRECTORY`, process the data using the `process_file` function, and aggregate the results into lists for further analysis.


In [4]:
# Lists to store processed data from each file
processed_O_data = []
processed_X_data = []

# Iterate over each file in the DATA_DIRECTORY
for filename in os.listdir(DATA_DIRECTORY):
    file_path = os.path.join(DATA_DIRECTORY, filename)
    
    if os.path.isfile(file_path):
        x_data, o_data = process_file(file_path, filename)
        
        if x_data is not None and o_data is not None:
            processed_X_data.append(x_data)
            processed_O_data.append(o_data)


Info: Non-integer value 'X' encountered in O data. Set to -1.
Info: Non-integer value 'X' encountered in O data. Set to -1.
Info: Non-integer value 'X' encountered in O data. Set to -1.
Info: Non-integer value 'X' encountered in O data. Set to -1.


## Converting Processed Data to NumPy Arrays

Convert the aggregated lists of processed data into NumPy arrays for efficient numerical operations.


In [5]:
# Convert lists to NumPy arrays
X = np.array(processed_X_data)
Y = np.array(processed_O_data)


## Data Validation

Ensure that the number of ground truth labels does not exceed the dimensions of the data arrays to prevent indexing errors during comparison.


In [6]:
# Validate that the number of labels matches the data dimensions
num_labels = len(GROUND_TRUTH_LABELS)
if X.shape[1] < num_labels or Y.shape[1] < num_labels:
    raise ValueError("Number of ground truth labels exceeds data dimensions.")


## Calculating Correct Predictions

Compare the first element of each triplet in `X` and `Y` against the corresponding ground truth labels to count the number of correct predictions.


In [7]:
# Calculate correct predictions for X and Y
correct_predictions_X = np.sum(X[:, :num_labels, 0] == GROUND_TRUTH_LABELS, axis=1).sum()
correct_predictions_Y = np.sum(Y[:, :num_labels, 0] == GROUND_TRUTH_LABELS, axis=1).sum()


## Calculating Word Error Rate (WER)

Compute the accuracy and subsequently the Word Error Rate for both `X` and `Y` datasets based on the correct predictions.


In [8]:
# Total number of predictions
total_predictions = num_labels * X.shape[0]

# Calculate accuracy
accuracy_X = correct_predictions_X / total_predictions
accuracy_Y = correct_predictions_Y / total_predictions

# Calculate Word Error Rate (WER)
WER_X = (1 - accuracy_X) * 100
WER_Y = (1 - accuracy_Y) * 100


## Results

Display the calculated Word Error Rates for both `X` and `Y` datasets.


In [9]:
# Print WER results
print(f"Word Error Rate for X: {WER_X:.2f}%")
print(f"Word Error Rate for Y: {WER_Y:.2f}%")


Word Error Rate for X: 16.80%
Word Error Rate for Y: 44.92%


Disclaimer: The documentation/comments of this code has been realized with the assistance of GPT/LLM tools but was not entirely written by them.